# Exploración aislada de exchange_rates.csv

Mi objetivo es conocer estructura, granularidad, calidad, cobertura y distribución de las tasas antes de proponer su uso. Analizo **únicamente `datos/exchange_rates.csv`**: no leo ni cruzo sales, inventory o ecommerce; tampoco consulto cotizaciones externas. No demuestro cobertura de pedidos ni exactitud de mercado.

Sigo la estructura del [informe de sales](../docs/02_exploracion/resultados_sales.md), adaptando sus controles a esta fuente.

## Documentación relacionada

- [Mis resultados de exchange_rates](../docs/02_exploracion/resultados_exchange_rates.md).
- [Índice de exploraciones](../docs/02_exploracion/README.md).
- [Preguntas al cliente](../docs/03_negocio/PREGUNTAS_CLIENTE.md#evidencia-del-eda-aislado-de-exchange-rates): D013, Q10 y Q38–Q40.
- [Dudas](../docs/03_negocio/DUDAS.md), [hallazgos](../docs/04_seguimiento/HALLAZGOS.md), [continuidad](../docs/04_seguimiento/CONTINUIDAD.md) y [bitácora de IA](../AI_LOG.md).

In [1]:
# Localizo la raíz desde las dos ubicaciones admitidas y preparo las herramientas del análisis aislado.
from pathlib import Path
import hashlib
import csv
from collections import Counter
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd() if (Path.cwd() / "datos").is_dir() else Path.cwd().parent
ARCHIVO = RAIZ / "datos" / "exchange_rates.csv"
assert ARCHIVO.is_file(), "Ejecuta desde la raíz del proyecto o desde notebooks/."

## 1. Lectura e identificación de la versión

Conservo las tres columnas como texto para distinguir vacíos, representaciones originales y precisión escrita. Creo las conversiones en columnas derivadas; no modifico la fuente.

In [2]:
# Identifico exactamente los bytes leídos y evito que pandas sustituya textos originales por nulos.
huella_inicial = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
original = pd.read_csv(ARCHIVO, dtype=str, keep_default_na=False)
columnas = ["fecha", "currency", "rate_to_mxn"]
assert original.columns.tolist() == columnas, "El esquema cambió; revisa la lectura."
fx = original.copy(deep=True)
fx["fecha_dt"] = pd.to_datetime(fx["fecha"], format="%Y-%m-%d", errors="coerce")
fx["tasa_num"] = pd.to_numeric(fx["rate_to_mxn"], errors="coerce")
display(pd.Series({"archivo": "datos/exchange_rates.csv", "bytes": ARCHIVO.stat().st_size, "sha256": huella_inicial, "filas": len(original), "columnas": len(original.columns)}, name="versión").to_frame())
display(original.head())

,versión
archivo,datos/exchange_rates.csv
bytes,16562
sha256,a0e827ec1e048cf397584eca3d692cb5b5b1f6b477fb78...
filas,730
columnas,3


,fecha,currency,rate_to_mxn
0,2025-04-01,USD,17.6958
1,2025-04-01,EUR,20.377
2,2025-04-02,USD,17.6524
3,2025-04-02,EUR,20.4385
4,2025-04-03,USD,17.6609


## 2. Esquema, diccionario y granularidad

Interpreto `fecha` como fecha registrada sin hora o zona explícitas; `currency` como código de moneda observado y `rate_to_mxn` como tasa declarada hacia MXN. El nombre sugiere MXN por unidad extranjera: es una hipótesis, no una convención confirmada. No hay proveedor, tipo de cotización, hora de corte ni vigencia explícita. Compruebo si fecha–moneda identifica una fila antes de tratar la fuente como serie diaria.

In [3]:
# Resumo las columnas originales y separo duplicados completos, claves repetidas y conflictos de tasa.
esquema = pd.DataFrame({"tipo_leído": original.dtypes.astype(str), "nulos": original.isna().sum(), "valores_distintos": original.nunique(dropna=False)})
tasas_por_clave = fx.groupby(["fecha", "currency"], dropna=False)["rate_to_mxn"].nunique()
display(esquema)
display(pd.Series({"duplicados_exactos_adicionales": int(original.duplicated().sum()), "claves_repetidas_adicionales": int(original.duplicated(["fecha", "currency"]).sum()), "claves_con_tasas_textuales_distintas": int(tasas_por_clave.gt(1).sum())}).to_frame("resultado"))
display(fx["currency"].value_counts().rename("registros").to_frame())

,tipo_leído,nulos,valores_distintos
fecha,object,0,365
currency,object,0,2
rate_to_mxn,object,0,649


,resultado
duplicados_exactos_adicionales,0
claves_repetidas_adicionales,0
claves_con_tasas_textuales_distintas,0


,registros
currency,
USD,365
EUR,365


## 3. Calidad estructural y numérica

Distingo texto vacío, tasa no interpretable, no finita, cero y negativa. Una ausencia de fila se evaluará contra el calendario, por separado. Estos controles detectan consistencia interna; no prueban que las cotizaciones sean reales.

In [4]:
# Evalúo formatos y valores sin corregir, imputar ni eliminar registros originales.
controles = pd.Series({
    "nulos_originales": int(original.isna().sum().sum()),
    "campos_vacíos_o_espacios": int(original.apply(lambda s: s.str.strip().eq("")).sum().sum()),
    "campos_con_espacios_extremos": int(original.apply(lambda s: s.ne(s.str.strip())).sum().sum()),
    "fecha_fuera_de_formato_ISO": int((~fx["fecha"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()),
    "fecha_no_interpretable": int(fx["fecha_dt"].isna().sum()),
    "moneda_fuera_de_tres_mayúsculas": int((~fx["currency"].str.fullmatch(r"[A-Z]{3}")).sum()),
    "tasa_no_interpretable": int(fx["tasa_num"].isna().sum()),
    "tasa_infinita": int(np.isinf(fx["tasa_num"]).sum()),
    "tasa_cero": int(fx["tasa_num"].eq(0).sum()),
    "tasa_negativa": int(fx["tasa_num"].lt(0).sum()),
})
display(controles.to_frame("registros_o_campos"))
assert controles.eq(0).all(), "Hay incidencias: revisa los controles antes de continuar."
assert not original.duplicated(["fecha", "currency"]).any(), "Resuelve la granularidad antes de calcular variaciones."

,registros_o_campos
nulos_originales,0
campos_vacíos_o_espacios,0
campos_con_espacios_extremos,0
fecha_fuera_de_formato_ISO,0
fecha_no_interpretable,0
moneda_fuera_de_tres_mayúsculas,0
tasa_no_interpretable,0
tasa_infinita,0
tasa_cero,0
tasa_negativa,0


## 4. Cobertura temporal

Construyo el calendario entre los extremos observados para cada moneda, sin asumir que representa todo el período requerido por el negocio. Incluyo fines de semana: no dispongo de calendario de publicación ni sé si son cotizaciones nuevas o tasas arrastradas.

In [5]:
# Comparo las claves observadas con el calendario diario esperado dentro de la ventana del archivo.
calendario = pd.date_range(fx["fecha_dt"].min(), fx["fecha_dt"].max(), freq="D")
monedas = sorted(fx["currency"].unique())
esperado = pd.MultiIndex.from_product([calendario, monedas], names=["fecha_dt", "currency"])
observado = pd.MultiIndex.from_frame(fx[["fecha_dt", "currency"]])
faltantes = esperado.difference(observado)
cobertura = fx.groupby("currency").agg(registros=("fecha", "size"), fechas_distintas=("fecha_dt", "nunique"), inicio=("fecha_dt", "min"), fin=("fecha_dt", "max"))
cobertura["días_esperados"] = len(calendario)
cobertura["días_sin_registro"] = cobertura["días_esperados"] - cobertura["fechas_distintas"]
fx["mes"] = fx["fecha_dt"].dt.to_period("M").astype(str)
mensual = fx.groupby(["mes", "currency"]).agg(registros=("fecha", "size"), mínima=("tasa_num", "min"), media=("tasa_num", "mean"), máxima=("tasa_num", "max"))
display(cobertura)
display(pd.Series({"combinaciones_esperadas": len(esperado), "combinaciones_ausentes": len(faltantes), "registros_en_fin_de_semana": int(fx["fecha_dt"].dt.dayofweek.ge(5).sum())}).to_frame("resultado"))
display(mensual.round(4))

,registros,fechas_distintas,inicio,fin,días_esperados,días_sin_registro
currency,,,,,,
EUR,365,365,2025-04-01,2026-03-31,365,0
USD,365,365,2025-04-01,2026-03-31,365,0


,resultado
combinaciones_esperadas,730
combinaciones_ausentes,0
registros_en_fin_de_semana,208


registros   mínima    media   máxima
mes     currency                                      
2025-04 EUR              30  20.3770  21.3139  21.9307
        USD              30  17.4587  17.6033  17.8280
2025-05 EUR              31  21.4006  21.7552  22.0000
        USD              31  17.1295  17.4591  17.7589
2025-06 EUR              30  21.4229  21.7383  22.0000
        USD              30  17.3362  17.6249  17.9426
2025-07 EUR              31  21.6939  21.9164  22.0000
        USD              31  17.5230  17.8787  18.0528
2025-08 EUR              31  21.5472  21.8520  22.0000
        USD              31  17.8575  17.9563  18.1183
2025-09 EUR              30  21.5935  21.8600  22.0000
        USD              30  17.8177  17.9621  18.0386
2025-10 EUR              31  21.7020  21.9473  22.0000
        USD              31  17.3344  17.7519  17.9528
2025-11 EUR              30  21.5927  21.8301  22.0000
        USD              30  17.3795  17.5508  17.8352
2025-12 EUR              31  21.4529  21.8366  22.0000
        USD              31  17.3286  17.6254  17.8727
2026-01 EUR              31  21.2281  21.6098  21.9280
        USD              31  17.4890  17.5959  17.8256
2026-02 EUR              28  21.1247  21.4507  21.9017
        USD              28  17.7425  18.2727  18.8253
2026-03 EUR              31  21.4549  21.8725  22.0000
        USD              31  18.3814  18.6261  18.8036

## 5. Distribución, precisión y extremos

Describo cada moneda por separado: no sumo ni promedio USD y EUR conjuntamente. Los decimales escritos no confirman la regla contable de redondeo. La media mensual describe tasas del calendario; no es una tasa ponderada por ventas.

In [6]:
# Obtengo percentiles y precisión textual para distinguir distribución y representación de la tasa.
distribucion = fx.groupby("currency")["tasa_num"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
fx["decimales_escritos"] = fx["rate_to_mxn"].map(lambda s: max(0, -Decimal(s).as_tuple().exponent))
precision = pd.crosstab(fx["currency"], fx["decimales_escritos"])
display(distribucion.round(6))
display(precision)
display(fx.loc[fx.groupby("currency")["tasa_num"].idxmin(), ["fecha", "currency", "rate_to_mxn"]])
display(fx.loc[fx.groupby("currency")["tasa_num"].idxmax(), ["fecha", "currency", "rate_to_mxn"]])

,count,mean,std,min,1%,25%,50%,75%,99%,max
currency,,,,,,,,,,
EUR,365.0,21.751705,0.275546,20.3770,20.615112,21.6400,21.8290,21.9575,22.000000,22.0000
USD,365.0,17.823456,0.360634,17.1295,17.288056,17.5585,17.7487,17.9625,18.796816,18.8253


decimales_escritos,1,2,3,4
currency,,,,
EUR,63,3,23,276
USD,0,4,29,332


,fecha,currency,rate_to_mxn
1,2025-04-01,EUR,20.377
102,2025-05-22,USD,17.1295


,fecha,currency,rate_to_mxn
99,2025-05-20,EUR,22.0
660,2026-02-25,USD,18.8253


## 6. Cambios diarios y valores repetidos

Ordeno por moneda y fecha antes de calcular diferencias y variaciones porcentuales. Dejo el primer cambio como desconocido, no como cero. Un cambio cero puede ser una cotización repetida; no es una tasa cero ni demuestra ausencia de actualización. Investigo especialmente las repeticiones del máximo observado sin declararlas errores.

In [7]:
# Calculo cambios solo entre días consecutivos de la misma moneda; no relleno huecos.
serie = fx.sort_values(["currency", "fecha_dt"]).copy()
grupos = serie.groupby("currency", sort=False)
consecutivo = grupos["fecha_dt"].diff().dt.days.eq(1)
serie["cambio"] = grupos["tasa_num"].diff().where(consecutivo)
serie["variación_pct"] = grupos["tasa_num"].pct_change(fill_method=None).mul(100).where(consecutivo)
cambios = serie.groupby("currency").agg(cambios_evaluados=("cambio", "count"), cambios_cero=("cambio", lambda s: int(s.eq(0).sum())), mayor_subida_pct=("variación_pct", "max"), mayor_bajada_pct=("variación_pct", "min"))
display(cambios.round(6))
display(serie.loc[serie["variación_pct"].abs().groupby(serie["currency"]).idxmax(), ["fecha", "currency", "rate_to_mxn", "variación_pct"]])

,cambios_evaluados,cambios_cero,mayor_subida_pct,mayor_bajada_pct
currency,,,,
EUR,364,33,1.847083,-1.338055
USD,364,0,1.240146,-1.357118


,fecha,currency,rate_to_mxn,variación_pct
15,2025-04-08,EUR,21.0026,1.847083
168,2025-06-24,USD,17.5899,-1.357118


In [8]:
# Identifico rachas del máximo por moneda y frecuencia exacta mediante decimales, sin inferir su causa.
serie["es_máximo"] = serie["tasa_num"].eq(serie.groupby("currency")["tasa_num"].transform("max"))
serie["racha"] = ((serie["currency"].ne(serie["currency"].shift())) | (serie["es_máximo"].ne(serie["es_máximo"].shift())) | (~consecutivo)).cumsum()
rachas = serie.loc[serie["es_máximo"]].groupby(["currency", "racha"]).agg(inicio=("fecha_dt", "min"), fin=("fecha_dt", "max"), días=("fecha_dt", "size"))
repeticiones = serie.groupby("currency").agg(máximo=("tasa_num", "max"), días_en_máximo=("es_máximo", "sum"), tasas_distintas=("tasa_num", "nunique"))
repeticiones["porcentaje_en_máximo"] = repeticiones["días_en_máximo"].div(serie.groupby("currency").size()).mul(100)
repeticiones["racha_máxima_días"] = rachas.groupby("currency")["días"].max()
display(repeticiones.round(4))
display(rachas.sort_values("días", ascending=False).head(10))
display(serie.groupby("currency")["rate_to_mxn"].apply(lambda s: s.map(Decimal).value_counts().head(5)).rename("frecuencia").to_frame())

,máximo,días_en_máximo,tasas_distintas,porcentaje_en_máximo,racha_máxima_días
currency,,,,,
EUR,22.0000,63,294,17.2603,6
USD,18.8253,1,355,0.2740,1


inicio        fin  días
currency racha                            
EUR      60    2026-03-24 2026-03-29     6
         18    2025-08-12 2025-08-15     4
         6     2025-06-06 2025-06-09     4
         34    2025-10-08 2025-10-11     4
         40    2025-10-28 2025-10-31     4
         28    2025-09-13 2025-09-15     3
         14    2025-07-26 2025-07-28     3
         38    2025-10-20 2025-10-22     3
         50    2025-12-11 2025-12-13     3
         16    2025-08-07 2025-08-08     2

frecuencia
currency                    
EUR      22.0             63
         21.7766           2
         21.9926           2
         21.622            2
         21.8733           2
USD      17.5727           2
         17.9463           2
         17.6316           2
         17.6238           2
         17.5546           2

## 7. Señales exploratorias por IQR

Uso cercas de 1.5×IQR por moneda sobre los niveles de tasa como señal descriptiva. Una serie temporal puede cambiar de nivel y generar señales sin errores; no elimino ni corrijo observaciones. No uso esta regla para certificar tasas de mercado.

In [9]:
# Señalo valores fuera de las cercas por moneda conservando todas las observaciones y mostrando sus fechas.
cuartiles = fx.groupby("currency")["tasa_num"].quantile([0.25, 0.75]).unstack()
cuartiles.columns = ["q1", "q3"]
cuartiles["iqr"] = cuartiles["q3"] - cuartiles["q1"]
cuartiles["límite_inferior"] = cuartiles["q1"] - 1.5 * cuartiles["iqr"]
cuartiles["límite_superior"] = cuartiles["q3"] + 1.5 * cuartiles["iqr"]
evaluado = fx.join(cuartiles, on="currency")
evaluado["señal_iqr"] = evaluado["tasa_num"].lt(evaluado["límite_inferior"]) | evaluado["tasa_num"].gt(evaluado["límite_superior"])
display(cuartiles.round(6))
display(evaluado.groupby("currency")["señal_iqr"].agg(["size", "sum"]))
display(evaluado.loc[evaluado["señal_iqr"], ["fecha", "currency", "rate_to_mxn"]])

,q1,q3,iqr,límite_inferior,límite_superior
currency,,,,,
EUR,21.6400,21.9575,0.3175,21.16375,22.43375
USD,17.5585,17.9625,0.4040,16.95250,18.56850


,size,sum
currency,,
EUR,365,11
USD,365,32


,fecha,currency,rate_to_mxn
1,2025-04-01,EUR,20.377
3,2025-04-02,EUR,20.4385
5,2025-04-03,EUR,20.6359
7,2025-04-04,EUR,20.6578
9,2025-04-05,EUR,20.6034
11,2025-04-06,EUR,20.5424
13,2025-04-07,EUR,20.6217
15,2025-04-08,EUR,21.0026
623,2026-02-06,EUR,21.1325
629,2026-02-09,EUR,21.1505


## 8. Controles de cierre y hallazgos calculados

Contrasto claves y recuentos con la biblioteca estándar; compruebo conservación de originales y hash al finalizar. Las comprobaciones las ejecuta la IA; mi revisión y las respuestas del cliente permanecen pendientes.

In [10]:
# Contrasto recuentos mediante CSV y Counter y compruebo que ni columnas originales ni bytes cambiaron.
with ARCHIVO.open(encoding="utf-8", newline="") as entrada:
    filas_csv = list(csv.DictReader(entrada))
claves_csv = Counter((f["fecha"], f["currency"]) for f in filas_csv)
assert len(filas_csv) == len(original) == int(mensual["registros"].sum())
assert len(claves_csv) == len(original) and max(claves_csv.values()) == 1
assert Counter(f["currency"] for f in filas_csv) == fx["currency"].value_counts().to_dict()
assert sum(Decimal(f["rate_to_mxn"]) == Decimal("22") for f in filas_csv if f["currency"] == "EUR") == int(((fx["currency"] == "EUR") & fx["tasa_num"].eq(22)).sum())
pd.testing.assert_frame_equal(original, fx[columnas])
assert hashlib.sha256(ARCHIVO.read_bytes()).hexdigest() == huella_inicial
resumen = pd.Series({"registros": len(fx), "monedas": ", ".join(monedas), "inicio": str(calendario.min().date()), "fin": str(calendario.max().date()), "días_calendario": len(calendario), "claves_ausentes": len(faltantes), "EUR_igual_a_22": int(((fx["currency"] == "EUR") & fx["tasa_num"].eq(22)).sum()), "fuente_intacta": True})
display(resumen.to_frame("resultado"))

,resultado
registros,730
monedas,"EUR, USD"
inicio,2025-04-01
fin,2026-03-31
días_calendario,365
claves_ausentes,0
EUR_igual_a_22,63
fuente_intacta,True


## 9. Dudas y próximos pasos

Conecto mis observaciones con [la evidencia de preguntas](../docs/03_negocio/PREGUNTAS_CLIENTE.md#evidencia-del-eda-aislado-de-exchange-rates):

- **D013 / Q10 y Q38:** confirmar MXN por unidad, proveedor y tipo de tasa (Q10), fecha aplicable y tratamiento de fines de semana (Q38). Mi cobertura interna no demuestra cobertura de transacciones.
- **D013 / Q39:** explicar las repeticiones de EUR en el máximo observado de 22.0; son compatibles con varias metodologías, ninguna confirmada.
- **D023 / Q40:** confirmar precisión, momento de redondeo y tolerancia. Los decimales escritos no especifican la política de cálculo.

No hay filas MXN: planteo identidad 1 únicamente como supuesto de conversión futura sujeto a confirmar, no como tasa faltante que deba imputar. No se envían preguntas ni se aplican conversiones de ventas.

## Conclusión

Tengo un perfil aislado de estructura, calendario, distribución y variaciones de esta fuente, con originales conservados y dudas vinculadas. Puedo preparar controles de integración, pero mantengo pendientes la convención de conversión, procedencia, redondeo y explicación de tasas repetidas antes de interpretar resultados de negocio.